# Notebook F2b: does the in-context count use the same machinery on POPE as on gratings, and as the learned prior? (GPU, about 85 min)

Notebook F (behaviour) found that the count sentence moves POPE answers only about +0.09 logits (gratings: +0.66), with no interaction with the split. This notebook asks the *mechanistic* version of Q6 with four tests. All use the same 180 POPE questions; no result is assumed.

| Test | Question | Reads as "same machinery" if | Reads as "separate" if |
|---|---|---|---|
| **M1 token-position patching** | Where is the count read on POPE: digits, sentence words, question tokens, or the answer position, and at which layers? | same chain as gratings (digits early, sentence words L8-14, answer position L16-20) | a different chain (other layers, other positions) |
| **M2 representation transfer** | Is the count direction measured on *gratings* also the direction in which the count moves LLaVA's state on *POPE*? (paired $k{=}8$ minus $k{=}2$ states; sign-flip nulls; also with the evidence axes projected out) | cosine and projection clearly beyond the null | inside the null |
| **M2b learned-prior axis** | Is the axis along which LLaVA's *image-free* yes-tendency varies (blank-image questions; the learned prior) aligned with the count direction? | cosine beyond a permutation null | inside it |
| **M3 steering transfer** | Does *adding* the gratings count direction (and the POPE count / learned-prior directions) shift POPE yes/no beyond what its overlap with the evidence axis predicts? (the B2 control: calibration directions, 6 shuffled nulls) | residual beyond the null spread | residual inside it |

**Important limits (stated before the run).** (1) The behavioural count effect on POPE is small, so M1 recoveries are fractions of a small gap; they are measured on the same question in two prompts (deterministic), so they are precise, but they describe a weak effect. (2) Gratings answer with the tokens Right/Left and POPE with Yes/No, so a failed transfer (M2/M3) does not by itself prove the mechanisms differ: the read-out differs. The notebook prints the cosine between the two read-out axes and between the two evidence axes so you can see how much transfer could be expected at all. (3) A shared direction would show a shared *representation*, not that the model *uses* it for hallucination; M3 is the causal check, with B2's controls.

**Setup:** GPU on (2x T4), **Internet on**, Kaggle secret `KAGGLE_KEY` on, *Save & Run All (Commit)*. No input dataset needed. Checkpoints go to `ponishbhatia3/llava-pve-f2b-ckpt` and resume on a re-run. Results are printed after each stage, so a partial run is still useful. `QUICK = True` is a smoke test (about 10 min).
**Tested on a mock model and on a tiny Llama (hooks, positions, steering, hidden states); not on the real LLaVA. Self-test cells stop with a clear message if something is off.**

In [ ]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~10 min)
SEED = 0
OUT = "/kaggle/working/F2b_out"
N_PER_SPLIT = 60                              # POPE questions per split (half present, half absent): 180 in total
N_PATCH = 72                                  # questions for token-position patching (stratified; multiple of 6)
N_STEER = 36                                  # questions for steering (subset of the patching questions; multiple of 6)
SL = [4, 8, 10, 12, 14, 16, 18, 20, 22, 24, 28]  # layers whose last-position state is stored
P_LAYERS = [4, 8, 12, 14, 16, 18, 20, 24]
P_SETS = ["digits", "words", "suffix", "last", "post_hist"]
STEER_L = [16, 20]
STEER_SIZE = 3.0                              # steering size = STEER_SIZE x |natural POPE count shift| at that layer
NNULL = 6                                     # shuffled/sign-flip null directions in the steering test
N_GRAT = 30                                   # gratings per side and per block (2 blocks: contrast 0.125 and 1.0)
NSIGN = 400                                   # sign-flip / permutation draws for the alignment nulls
KS = (2, 8)
NEUTRAL = "The picture was taken with a plain camera in a quiet room on a sunny afternoon."
if QUICK:
    N_PER_SPLIT, N_PATCH, N_STEER, P_LAYERS, STEER_L, NNULL, N_GRAT, NSIGN = 12, 12, 6, [8, 14, 20], [20], 2, 6, 100
    SL = [8, 14, 20]
assert set(STEER_L) <= set(SL) and N_PATCH % 6 == 0 and N_STEER % 6 == 0 and N_STEER <= N_PATCH
import os, re, glob, json, math, time, zipfile, shutil, warnings, subprocess, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [ ]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-f2b-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/F2b_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_PER_SPLIT=N_PER_SPLIT, N_PATCH=N_PATCH, N_STEER=N_STEER, SL=SL, P_LAYERS=P_LAYERS, P_SETS=P_SETS, STEER_L=STEER_L, STEER_SIZE=STEER_SIZE, NNULL=NNULL, N_GRAT=N_GRAT)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook F2b (mechanism of the POPE count effect). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json", "*.npz"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE F2b checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("states.npz", "tokpatch_rows.csv", "steer_rows.csv"):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

In [ ]:
# ===================== POPE questions (stratified) and gratings stimuli =====================
class FakeImg:                                   # mock only
    def __init__(s, **kw): s.__dict__.update(kw)
def load_pope():
    if MOCK:
        rows = []
        for sp in ("random", "popular", "adversarial"):
            for i in range(N_PER_SPLIT):
                rows.append(dict(qid=f"{sp}{i}", split=sp, truth=int(i % 2), question=f"Is there a thing{i % 7} in the image?", img_id=f"img{i % 40}"))
        df = pd.DataFrame(rows); df["image"] = [FakeImg(kind="pope", truth=t, split=s, i=k, blank=False) for k, (t, s) in enumerate(zip(df.truth, df.split))]; return df
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("lmms-lab/POPE")
    if hasattr(ds, "keys"): ds = concatenate_datasets([ds[k] for k in ds.keys()])
    cols = ds.column_names; print("POPE columns:", cols, "| rows:", len(ds))
    def pick(*names):
        for n in names:
            if n in cols: return n
        raise AssertionError(f"POPE loader: none of {names} found in columns {cols}. Send me this message.")
    cq, ca, cc, ci = pick("question"), pick("answer", "label"), pick("category", "split", "type"), pick("image")
    cid = next((n for n in ("image_source", "image_id", "image_name") if n in cols), None)
    meta = ds.remove_columns([ci]).to_pandas(); meta["_row"] = np.arange(len(meta))
    meta["truth"] = meta[ca].astype(str).str.lower().str.strip().map({"yes": 1, "no": 0}); assert meta.truth.notna().all()
    meta["split"] = meta[cc].astype(str).str.lower(); meta["img_id"] = meta[cid].astype(str) if cid else meta["_row"].astype(str)
    meta["question"] = meta[cq]; meta["qid"] = meta["_row"].astype(str)
    rng = np.random.default_rng(SEED); keep = []
    for sp in ("random", "popular", "adversarial"):
        for t in (1, 0):
            pool = meta[(meta.split == sp) & (meta.truth == t)]._row.values; assert len(pool) >= N_PER_SPLIT // 2
            keep += list(rng.choice(pool, N_PER_SPLIT // 2, replace=False))
    sub = meta[meta._row.isin(keep)].copy(); sub["image"] = [ds[int(r)][ci].convert("RGB") for r in sub._row]
    return sub[["qid", "split", "truth", "question", "img_id", "image"]].reset_index(drop=True)
Q = load_pope(); assert len(Q) == 3 * N_PER_SPLIT; print(Q.groupby(["split", "truth"]).size().unstack())
# fixed random order inside each (split, truth) cell: the patching set is the first N_PATCH/6 of each cell, the steering set the first N_STEER/6
rq = np.random.default_rng(SEED + 5); order = {}
for (sp, t), g in Q.groupby(["split", "truth"]): order[(sp, t)] = list(rq.permutation(g.index.values))
def subset(n): return sorted(sum([order[k][: n // 6] for k in order], []))
PATCH_Q, STEER_Q = subset(N_PATCH), subset(N_STEER)
BLANK_CACHE = {}
def blank_img(i):
    if MOCK: return FakeImg(kind="pope", truth=0, split=Q.split.iloc[i], i=i, blank=True)
    if "g" not in BLANK_CACHE: BLANK_CACHE["g"] = Image.new("RGB", (336, 336), (128, 128, 128))
    return BLANK_CACHE["g"]
# ---- gratings (same renderer and wording as the main run): block 0 = contrast 0.125 (count direction), block 1 = contrast 1.0 (evidence axis)
SIZE = 336; _Y, _X = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)
rg_ = np.random.default_rng(SEED + 11); NG = 4 * N_GRAT
GT = dict(side=np.array((["right"] * N_GRAT + ["left"] * N_GRAT) * 2), contrast=np.r_[np.full(2 * N_GRAT, 0.125), np.full(2 * N_GRAT, 1.0)], dx=rg_.uniform(-15, 15, NG), dy=rg_.uniform(-30, 30, NG), phi=rg_.uniform(0, 2 * np.pi, NG))
def render(t):
    if MOCK: return FakeImg(kind="grat", sgn=1 if GT["side"][t] == "right" else -1, i=t, blank=False)
    cx = SIZE / 2 + (84 if GT["side"][t] == "right" else -84) + GT["dx"][t]; cy = SIZE / 2 + GT["dy"][t]
    env = np.exp(-((_X - cx) ** 2 + (_Y - cy) ** 2) / (2 * 36 ** 2)); a = 0.5 + 0.5 * GT["contrast"][t] * env * np.sin(2 * np.pi * (_X - cx) / 24 + GT["phi"][t])
    a = a + np.random.default_rng(SEED * 100003 + t).normal(0, 0.02, a.shape)
    return Image.fromarray(np.stack([(np.clip(a, 0, 1) * 255).astype(np.uint8)] * 3, -1))
GQ = "Is the grating on the left or right of the image? Answer with one word."

In [ ]:
# ===================== model runner =====================
D = 4096
if MOCK:
    D = 48; NLAY = 33; _r = np.random.default_rng(123)
    def _u(v): return v / np.linalg.norm(v)
    EVg, CG = _u(_r.normal(size=D)), _u(_r.normal(size=D)); EVp = _u(_r.normal(size=D)); CP = _u(0.6 * CG + 0.25 * _r.normal(size=D) / np.sqrt(D) * 3 + 0.25 * EVp); PRI = _u(_r.normal(size=D))
    WP, WG = 2.0 * EVp, 2.0 * EVg
    class Runner:
        def prep(s, img, text):
            m = re.search(r"answer was yes (\d+) times", text) or re.search(r"on the right (\d+) times", text)
            return dict(img=img, k=int(m.group(1)) if m else None, neutral=NEUTRAL in text, text=text, kind=img.kind)
        def _state(s, inp, L):
            im = inp["img"]; a = min(1.0, L / 16)
            if inp["kind"] == "pope":
                rr = np.random.default_rng(im.i); prior = rr.normal(); sg = 0 if im.blank else (2 * im.truth - 1) * 1.5
                x = sg * EVp + prior * 0.8 * PRI + 0.3 * rr.normal(size=D) / np.sqrt(D) * 4
                if inp["k"] is not None: x = 0.5 * x + (inp["k"] - 5) * 0.08 * CP
                return a * x
            rr = np.random.default_rng(1000 + im.i); x = im.sgn * 1.5 * (1.0 if "right (" not in inp["text"] else 1.0) * EVg + 0.3 * rr.normal(size=D) / np.sqrt(D) * 4
            if inp["k"] is not None: x = 0.5 * x + (inp["k"] - 5) * 0.12 * CG
            return a * x
        def fwd(s, inp, layers=None, edit=None, pair="yn"):
            x = s._state(inp, 32); w = WP if inp["kind"] == "pope" else WG
            if edit is not None: x = x + edit[1]
            d = float(w @ x + (0.0 if inp["kind"] == "pope" else 0.0))
            return d, (None if layers is None else np.stack([s._state(inp, L) for L in layers]).astype(np.float16))
        def delta(s, inp, edit=None, pair="yn"): return s.fwd(inp, None, edit, pair)[0]
        def readout_axis(s, pair="yn"): return (WP if pair == "yn" else WG).astype(np.float32)
        def positions(s, ia, ib, inone): return dict(digits=["d"], words=["w"], suffix=["s"], last=["l"], post_hist=["d", "w", "s", "l"])
        def capture(s, inp, Ls, pos): return {"inp": inp}
        def delta_pos(s, inp, L, pos, cap, rowmap):
            f = {"d": 1.0 if L <= 10 else 0.1, "w": 0.3 if L <= 14 else 0.0, "s": 0.1, "l": 0.0 if L < 14 else min(1.0, (L - 14) / 6)}
            fr = 1.0 if len(pos) > 1 else f[pos[0]]; a, b = s.delta(inp), s.delta(cap["inp"]); return a + fr * (b - a)
    RUN = Runner()
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto", quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    NO_ID, YES_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("No", "Yes")]
    L_ID, R_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("Left", "Right")]
    print("token ids No/Yes/Left/Right:", NO_ID, YES_ID, L_ID, R_ID, [proc.tokenizer.convert_ids_to_tokens(i) for i in (NO_ID, YES_ID, L_ID, R_ID)])
    _lists = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == 32]
    assert len(_lists) == 1, f"expected exactly one 32-layer list, found {[n for n, _ in _lists]}"
    LAYERS = _lists[0][1]; print("language-model layers found at:", _lists[0][0])
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def _ids(s, pair): return (YES_ID, NO_ID) if pair == "yn" else (R_ID, L_ID)
        def fwd(s, inp, layers=None, edit=None, pair="yn"):
            handle = None
            if edit is not None:
                L, vec = edit; v0 = torch.tensor(vec, dtype=torch.float16)
                def hook(mod, args, output):                      # output of decoder layer L-1 == hidden_states[L]; add at the last position
                    h = output[0] if isinstance(output, tuple) else output
                    h = h.clone(); h[:, -1, :] = h[:, -1, :] + v0.to(h.device, h.dtype)
                    return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
                handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): out = model(**inp, output_hidden_states=layers is not None)
            finally:
                if handle is not None: handle.remove()
            a, b = s._ids(pair); lg = out.logits[0, -1].float(); d = float((lg[a] - lg[b]).item()); st = None
            if layers is not None: st = np.stack([out.hidden_states[L][0, -1].float().cpu().numpy() for L in layers]).astype(np.float16)
            return d, st
        def delta(s, inp, edit=None, pair="yn"): return s.fwd(inp, None, edit, pair)[0]
        def readout_axis(s, pair="yn"):
            a, b = s._ids(pair); W = model.lm_head.weight.detach(); w = (W[a] - W[b]).float().cpu().numpy()
            gl = [m for n, m in model.named_modules() if n.endswith("language_model.norm") or n == "model.norm"]
            g = gl[0].weight.detach().float().cpu().numpy() if gl else np.ones_like(w); return (g * w).astype(np.float32)
        def positions(s, ia, ib, inone):
            a, b, c = [x["input_ids"][0].cpu().numpy() for x in (ia, ib, inone)]
            assert len(a) == len(b), "donor and recipient prompts differ in token length"
            IMG = getattr(model.config, "image_token_index", None) or getattr(model.config, "image_token_id", None) or proc.tokenizer.convert_tokens_to_ids("<image>")
            img = np.where(a == IMG)[0]; assert len(img) > 100, "image tokens not expanded in input_ids: position bookkeeping would be wrong"
            n, h0 = len(a), int(img.max()) + 1; suf = 0
            while suf < len(c) - 1 and a[n - 1 - suf] == c[len(c) - 1 - suf]: suf += 1
            dig = np.where(a != b)[0].tolist(); region = list(range(h0, n - suf))
            return dict(digits=dig, words=[p for p in region if p not in dig], suffix=list(range(n - suf, n - 1)), last=[n - 1], post_hist=list(range(h0, n)))
        def capture(s, inp, Ls, pos):
            with torch.no_grad(): out = model(**inp, output_hidden_states=True)
            assert out.hidden_states[0].shape[1] == inp["input_ids"].shape[1], "hidden-state length != input_ids length"
            idx = torch.tensor(pos)
            return {L: out.hidden_states[L][0].index_select(0, idx.to(out.hidden_states[L].device)).cpu() for L in Ls}
        def delta_pos(s, inp, L, pos, cap, rowmap):
            rows = torch.tensor([rowmap[p] for p in pos]); v0 = cap[L].index_select(0, rows); pidx = torch.tensor(pos)
            def hook(mod, args, output):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone(); h[:, pidx.to(h.device), :] = v0.to(h.device, h.dtype)
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handle = LAYERS[L - 1].register_forward_hook(hook)
            try:
                with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            finally: handle.remove()
            return float((lg[YES_ID] - lg[NO_ID]).item())
    RUN = Runner()
def pprompt(hist, q):
    core = f"{q}\nAnswer the question using a single word or phrase."
    return f"USER: <image>\n{hist}\n{core} ASSISTANT:" if hist else f"USER: <image>\n{core} ASSISTANT:"
def phist(k): return f"In the last 10 questions the answer was yes {k} times and no {10 - k} times."
def gprompt(hist): return f"USER: <image>\n{hist}\n{GQ} ASSISTANT:" if hist else f"USER: <image>\n{GQ} ASSISTANT:"
def ghist(k): return f"In the last 10 trials the grating was on the left {10 - k} times and on the right {k} times."

In [ ]:
# ===================== SELF-TESTS =====================
r0 = Q.iloc[0]; ip0, ip2, ip8 = [RUN.prep(r0.image, pprompt(h, r0.question)) for h in (None, phist(2), phist(8))]
d0, st0 = RUN.fwd(ip0, [SL[-1]]); say(f"[self-test 1] first question '{r0.question}' truth={r0.truth}: delta(Yes-No) none {d0:+.2f} | k2 {RUN.delta(ip2):+.2f} | k8 {RUN.delta(ip8):+.2f}")
if not MOCK:
    top = proc.tokenizer.convert_ids_to_tokens(int(model(**ip0).logits[0, -1].argmax())); assert top.replace("▁", "").strip().lower() in ("yes", "no"), f"first token is {top!r}, not Yes/No"
    ig = RUN.prep(render(0), gprompt(None)); topg = proc.tokenizer.convert_ids_to_tokens(int(model(**ig).logits[0, -1].argmax())); say(f"[self-test 2] gratings prompt starts with {topg!r} (should be Left/Right)"); assert topg.replace("▁", "").strip().lower() in ("left", "right")
    # stored state at layer L == what the hook sees at layer L (hidden_states[L] is the output of LAYERS[L-1])
    box = {}
    def spy(mod, args, output): box["h"] = (output[0] if isinstance(output, tuple) else output)[0, -1].float().cpu().numpy()
    hh = LAYERS[19].register_forward_hook(spy); _, s20 = RUN.fwd(ip0, [20]); hh.remove()
    err = np.abs(box["h"] - s20[0].astype(np.float32)).max(); say(f"[self-test 3] max |hook state - hidden_states[20]| = {err:.4f} (should be ~0)"); assert err < 0.05, "layer indexing is off"
z0 = RUN.delta(ip0, edit=(20, np.zeros(D, np.float32))); say(f"[self-test 4] adding a zero vector changes delta by {abs(z0 - RUN.delta(ip0)):.4f} (should be ~0)"); assert abs(z0 - RUN.delta(ip0)) < 0.05
w_ = RUN.readout_axis("yn"); u_ = w_ / np.linalg.norm(w_); big = RUN.delta(ip0, edit=(20, (30 * u_).astype(np.float32))) - RUN.delta(ip0)
say(f"[self-test 5] adding 30 x the Yes-No read-out axis at layer 20 moves delta by {big:+.2f} (should be clearly positive)"); assert big > 0.5, "steering hook or read-out axis wrong"
P0 = RUN.positions(ip2, ip8, ip0); say(f"[self-test 6] positions: digits {P0['digits']} | words {len(P0['words'])} | suffix {len(P0['suffix'])} | last {P0['last']} | post_hist {len(P0['post_hist'])}")
if not MOCK:
    ids = ip8["input_ids"][0].cpu().numpy(); say("   digit tokens: " + str([proc.tokenizer.decode([int(ids[p])]) for p in P0["digits"]]) + " | first words: " + str([proc.tokenizer.decode([int(ids[p])]) for p in P0["words"][:6]]) + " | first suffix tokens: " + str([proc.tokenizer.decode([int(ids[p])]) for p in P0["suffix"][:6]]))
    assert 1 <= len(P0["digits"]) <= 3 and 8 <= len(P0["words"]) <= 60 and set(P0["digits"]) <= set(P0["post_hist"]) and P0["last"][0] == P0["post_hist"][-1]
allp = P0["post_hist"]; rm0 = {p: j for j, p in enumerate(allp)}; d2_, d8_ = RUN.delta(ip2), RUN.delta(ip8); cap8 = RUN.capture(ip8, [4], allp)
dp_ = RUN.delta_pos(ip2, 4, allp, cap8, rm0); say(f"[self-test 7] overwriting ALL post-image positions at layer 4 of the k=2 run with the k=8 run: delta {dp_:+.3f} should equal the k=8 delta {d8_:+.3f}"); assert abs(dp_ - d8_) < (0.01 if MOCK else 0.15)
say("self-tests passed")

In [ ]:
# ===================== STAGE S: last-position states (POPE: no history / k=2 / k=8 / blank image; gratings: k=2 / k=8 at low contrast, no history at high contrast) =====================
sp_ = f"{OUT}/states.npz"; nq = len(Q)
if os.path.exists(sp_):
    z = np.load(sp_, allow_pickle=True); S = {k: z[k] for k in z.files}; print("states loaded from checkpoint")
else:
    S = {f"p_{p}": np.zeros((nq, len(SL), D), np.float16) for p in ("none", "k2", "k8", "blank")}; S.update({f"pd_{p}": np.zeros(nq) for p in ("none", "k2", "k8", "blank")})
    t0 = time.time()
    for i in range(nq):
        r = Q.iloc[i]
        for p, (img, txt) in dict(none=(r.image, pprompt(None, r.question)), k2=(r.image, pprompt(phist(2), r.question)), k8=(r.image, pprompt(phist(8), r.question)), blank=(blank_img(i), pprompt(None, r.question))).items():
            d, st = RUN.fwd(RUN.prep(img, txt), SL); S[f"p_{p}"][i] = st; S[f"pd_{p}"][i] = d
        if (i + 1) % 30 == 0: print(f"  POPE states {i + 1}/{nq} | {(time.time() - t0) / 60:.1f} min", flush=True)
    low = np.where(GT["contrast"] < 0.5)[0]; hi = np.where(GT["contrast"] >= 0.5)[0]
    for nm in ("k2", "k8"): S[f"g_{nm}"] = np.zeros((len(low), len(SL), D), np.float16); S[f"gd_{nm}"] = np.zeros(len(low))
    S["g_none_hi"] = np.zeros((len(hi), len(SL), D), np.float16); S["gd_none_hi"] = np.zeros(len(hi)); S["g_side_low"] = (GT["side"][low] == "right").astype(int); S["g_side_hi"] = (GT["side"][hi] == "right").astype(int)
    for j, t in enumerate(low):
        img = render(t)
        for nm, k in (("k2", 2), ("k8", 8)): d, st = RUN.fwd(RUN.prep(img, gprompt(ghist(k))), SL, pair="rl"); S[f"g_{nm}"][j] = st; S[f"gd_{nm}"][j] = d
    for j, t in enumerate(hi): d, st = RUN.fwd(RUN.prep(render(t), gprompt(None)), SL, pair="rl"); S["g_none_hi"][j] = st; S["gd_none_hi"][j] = d
    np.savez_compressed(sp_, **S); maybe_push("states", force=True); print(f"states done in {(time.time() - t0) / 60:.1f} min")
say(f"\n[S] POPE mean delta: none {S['pd_none'].mean():+.2f} | k2 {S['pd_k2'].mean():+.2f} | k8 {S['pd_k8'].mean():+.2f} | blank image {S['pd_blank'].mean():+.2f}; count effect k8-k2 = {(S['pd_k8'] - S['pd_k2']).mean():+.3f}")
say(f"    gratings (contrast 0.125): count effect delta(8 right)-delta(2 right) = {(S['gd_k8'] - S['gd_k2']).mean():+.3f} (Right-Left logits); evidence at contrast 1.0: right minus left = {S['gd_none_hi'][S['g_side_hi'] == 1].mean() - S['gd_none_hi'][S['g_side_hi'] == 0].mean():+.2f}")

In [ ]:
# ===================== M2: do the gratings count direction and the POPE count direction coincide? =====================
def unit(v): return v / (np.linalg.norm(v) + 1e-12)
def cosv(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
def orth(v, axes):
    if not len(axes): return v
    B, _ = np.linalg.qr(np.stack(axes, 1)); return v - B @ (B.T @ v)
rs = np.random.default_rng(7)
pres = Q.truth.values == 1; AX = {}; rows = []
tert_hi, tert_lo = np.quantile(S["pd_blank"], 2 / 3), np.quantile(S["pd_blank"], 1 / 3); gh, gl = S["pd_blank"] >= tert_hi, S["pd_blank"] <= tert_lo
for li, L in enumerate(SL):
    dg = (S["g_k8"][:, li].astype(np.float32) - S["g_k2"][:, li].astype(np.float32)); dp = (S["p_k8"][:, li].astype(np.float32) - S["p_k2"][:, li].astype(np.float32))
    Gv, Pv = dg.mean(0), dp.mean(0)
    Eg = S["g_none_hi"][S["g_side_hi"] == 1, li].astype(np.float32).mean(0) - S["g_none_hi"][S["g_side_hi"] == 0, li].astype(np.float32).mean(0)
    Hn = S["p_none"][:, li].astype(np.float32); Ep = Hn[pres].mean(0) - Hn[~pres].mean(0)
    Hb = S["p_blank"][:, li].astype(np.float32); Pr = Hb[gh].mean(0) - Hb[gl].mean(0)
    flips = rs.choice([-1.0, 1.0], size=(NSIGN, len(dg))); Gn = flips @ dg / len(dg)
    perm = [rs.permutation(len(gh)) for _ in range(NSIGN)]; Prn = np.stack([Hb[gh[p]].mean(0) - Hb[gl[p]].mean(0) for p in perm])
    c_gp = cosv(Gv, Pv); n_gp = np.array([cosv(g, Pv) for g in Gn])
    Gp_, Pp_ = orth(Gv, [Eg, Ep]), orth(Pv, [Ep]); n_gpp = np.array([cosv(orth(g, [Eg, Ep]), Pp_) for g in Gn]); c_gpp = cosv(Gp_, Pp_)
    c_pr = cosv(Pr, Pv); n_pr = np.array([cosv(p_, Pv) for p_ in Prn])
    half = lambda M: np.mean([cosv(M[a].mean(0), M[~a].mean(0)) for a in [(lambda k: np.isin(np.arange(len(M)), k))(rs.choice(len(M), len(M) // 2, replace=False)) for _ in range(30)]])
    rows.append(dict(layer=L, cos_G_P=c_gp, null95=np.quantile(np.abs(n_gp), .95), p_G_P=float((np.abs(n_gp) >= abs(c_gp)).mean()), size_ratio=np.linalg.norm(Pv) / np.linalg.norm(Gv),
                     cos_Gperp_Pperp=c_gpp, p_perp=float((np.abs(n_gpp) >= abs(c_gpp)).mean()), cos_prior_P=c_pr, p_prior=float((np.abs(n_pr) >= abs(c_pr)).mean()),
                     cos_Eg_Ep=cosv(Eg, Ep), cos_P_Ep=cosv(Pv, Ep), cos_G_Eg=cosv(Gv, Eg), cos_prior_Ep=cosv(Pr, Ep), reliab_P=half(dp), reliab_G=half(dg)))
    AX[L] = dict(G=Gv, P=Pv, Eg=Eg, Ep=Ep, Pr=Pr, Gn=Gn[:max(NNULL, 1)], Gp=Gp_, Pp=Pp_)
M2 = pd.DataFrame(rows); M2.to_csv(f"{OUT}/m2_alignment.csv", index=False)
say("\n[M2] alignment of count directions (last-position state, per layer). cos_G_P: gratings count direction vs POPE count direction; null95: 95th percentile of |cos| for sign-flip nulls built the same way from the gratings pairs; p: share of nulls at least as aligned (two-sided)")
say(M2[["layer", "cos_G_P", "null95", "p_G_P", "size_ratio", "cos_Gperp_Pperp", "p_perp", "reliab_P", "reliab_G"]].round(3).to_string(index=False))
say("(size_ratio = |POPE count shift| / |gratings count shift| of the state; reliab_* = split-half cosine of the direction with itself: the ceiling for any alignment that involves a noisy direction)")
say("\n[M2b] learned-prior axis (image-free: top vs bottom third of blank-image delta) vs POPE count direction; permutation null")
say(M2[["layer", "cos_prior_P", "p_prior", "cos_prior_Ep", "cos_P_Ep"]].round(3).to_string(index=False))
say("\n[M2c] calibration: do the two EVIDENCE axes (grating right-vs-left, object present-vs-absent) align across tasks, and do the count directions lean on them?")
say(M2[["layer", "cos_Eg_Ep", "cos_G_Eg", "cos_P_Ep"]].round(3).to_string(index=False))
wg, wp = RUN.readout_axis("rl"), RUN.readout_axis("yn"); say(f"   cosine between the two read-out axes (Right-Left vs Yes-No, unembedding x final-norm gain): {cosv(wg, wp):+.3f}")
# validity of the learned-prior proxy: does the blank-image delta predict the real-image delta on ABSENT objects, and does the image-free prior axis?
ab = ~pres
from scipy.stats import spearmanr
say(f"\n[M2d] validity of the image-free prior proxy on absent objects: Spearman(blank delta, real-image delta) = {spearmanr(S['pd_blank'][ab], S['pd_none'][ab])[0]:+.2f}; on present objects {spearmanr(S['pd_blank'][pres], S['pd_none'][pres])[0]:+.2f}")
for L in [l for l in (16, 20, 24) if l in SL]:
    li = SL.index(L); pj = S["p_none"][:, li].astype(np.float32) @ unit(AX[L]["Pr"]); say(f"   layer {L}: Spearman(real-image state projected on the image-free prior axis, real-image delta) on absent objects = {spearmanr(pj[ab], S['pd_none'][ab])[0]:+.2f}")

In [ ]:
# ===================== M1: token-position patching of the POPE count effect (checkpointed) =====================
tp = f"{OUT}/tokpatch_rows.csv"; tdone = pd.read_csv(tp) if os.path.exists(tp) else pd.DataFrame()
trows = tdone.to_dict("records") if len(tdone) else []; have_t = set(tdone.qi) if len(tdone) else set()
todo = [i for i in PATCH_Q if i not in have_t]; t0 = time.time()
print(f"token patching: {len(PATCH_Q)} questions x {len(P_SETS)} position sets x {len(P_LAYERS)} layers; questions left: {len(todo)}")
for n_, i in enumerate(todo):
    r = Q.iloc[i]; i2, i8, i0 = [RUN.prep(r.image, pprompt(h, r.question)) for h in (phist(2), phist(8), None)]
    P = RUN.positions(i2, i8, i0); allp = P["post_hist"]; rm = {p: j for j, p in enumerate(allp)}; d2, d8 = RUN.delta(i2), RUN.delta(i8); cap8 = RUN.capture(i8, P_LAYERS, allp)
    for L in P_LAYERS:
        for st in P_SETS:
            if len(P[st]) == 0: continue
            trows.append(dict(qi=i, split=r.split, truth=int(r.truth), set=st, layer=L, d_recip=d2, d_donor=d8, delta=RUN.delta_pos(i2, L, P[st], cap8, rm)))
    pd.DataFrame(trows).to_csv(tp, index=False); maybe_push("tokpatch"); el = time.time() - t0
    if (n_ + 1) % 4 == 0 or n_ == len(todo) - 1: print(f"  {n_ + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (n_ + 1) * (len(todo) - n_ - 1) / 60:.0f} min left", flush=True)
TP = pd.DataFrame(trows); maybe_push("tokpatch done", force=True)
g0 = TP[(TP.set == P_SETS[0]) & (TP.layer == P_LAYERS[0])]; gap = (g0.d_donor - g0.d_recip).values
say(f"\n[M1] delta(k=8) - delta(k=2) over {len(gap)} questions: {gap.mean():+.3f} (sd across questions {gap.std():.3f}); patching recovery below is relative to this gap")
rb = np.random.default_rng(0); rows = []
for st in P_SETS:
    for L in P_LAYERS:
        x = TP[(TP.set == st) & (TP.layer == L)]; num = (x.delta - x.d_recip).values; den = (x.d_donor - x.d_recip).values; rec = num.mean() / den.mean(); bs = []
        for _ in range(500): ii = rb.integers(0, len(num), len(num)); bs.append(num[ii].mean() / den[ii].mean())
        rows.append(dict(set=st, layer=L, recovery=rec, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
M1 = pd.DataFrame(rows); M1.to_csv(f"{OUT}/m1_tokpatch_summary.csv", index=False)
say("recovery = fraction of the k=2 vs k=8 gap that moves when that set of positions is overwritten at layer L (k=8 state into a k=2 run)")
say(M1.pivot(index="layer", columns="set", values="recovery")[P_SETS].round(2).to_string())
say("   [reference, gratings (B2): digits 0.60/0.18/~0 at L4/L8/L16+; sentence words 0.74/0.86/0.65/0.48 at L8/10/12/14; answer position 0.30/0.73/0.88 at L16/18/20; all post-image positions 1.0]")
say("   by split (recovery at L12 for the sentence words and at L20 for the answer position, if those layers were run):")
for L, st in ((12, "words"), (20, "last")):
    if L not in P_LAYERS: continue
    out = []
    for sp in ("random", "popular", "adversarial"):
        x = TP[(TP.set == st) & (TP.layer == L) & (TP.split == sp)]; out.append(f"{sp} {(x.delta - x.d_recip).mean() / (x.d_donor - x.d_recip).mean():.2f}")
    say(f"   {st} L{L}: " + " | ".join(out))
fig, ax = plt.subplots(figsize=(6.5, 4))
for st in P_SETS: x = M1[M1.set == st]; ax.errorbar(x.layer, x.recovery, yerr=[x.recovery - x.lo, x.hi - x.recovery], marker="o", label=st, capsize=2)
ax.axhline(1, c="gray", lw=.5); ax.axhline(0, c="gray", lw=.5); ax.set_xlabel("layer"); ax.set_title("M1: recovery of the POPE count effect"); ax.legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/m1_tokpatch.png", dpi=130); plt.close()

In [ ]:
# ===================== M3: steering transfer with B2's controls (checkpointed) =====================
rr = np.random.default_rng(3); DIRS = {}                                    # per layer: name -> unit direction
for L in STEER_L:
    A = AX[L]; Ephat = unit(A["Ep"]); d = {}
    d["Ep"] = Ephat
    for c in (0.15, 0.3, 0.5):
        r_ = orth(rr.normal(size=D).astype(np.float32), [Ephat]); d[f"cal{c}"] = unit(c * Ephat + np.sqrt(1 - c * c) * unit(r_))
    for j in range(NNULL): d[f"null{j + 1}"] = unit(A["Gn"][j])
    d["G"] = unit(A["G"]); d["Gperp"] = unit(orth(A["G"], [A["Eg"], A["Ep"]])); d["P"] = unit(A["P"]); d["Pperp"] = unit(orth(A["P"], [A["Ep"]])); d["Prior"] = unit(A["Pr"]); d["Priorperp"] = unit(orth(A["Pr"], [A["Ep"]]))
    DIRS[L] = d
SIZE_L = {L: STEER_SIZE * float(np.linalg.norm(AX[L]["P"])) for L in STEER_L}
sp2 = f"{OUT}/steer_rows.csv"; sdone = pd.read_csv(sp2) if os.path.exists(sp2) else pd.DataFrame()
srows = sdone.to_dict("records") if len(sdone) else []; have_s = set(zip(sdone.qi, sdone.layer)) if len(sdone) else set()
todo = [(i, L) for i in STEER_Q for L in STEER_L if (i, L) not in have_s]; t0 = time.time()
print(f"steering: {len(STEER_Q)} questions x {len(STEER_L)} layers x {len(DIRS[STEER_L[0]])} directions x 2 signs; sizes {SIZE_L}; left: {len(todo)} question-layers")
for n_, (i, L) in enumerate(todo):
    r = Q.iloc[i]; inp = RUN.prep(r.image, pprompt(None, r.question)); base = RUN.delta(inp)
    srows.append(dict(qi=i, split=r.split, truth=int(r.truth), layer=L, dir="base", sign=0, delta=base))
    for nm, u in DIRS[L].items():
        for sg in (1, -1): srows.append(dict(qi=i, split=r.split, truth=int(r.truth), layer=L, dir=nm, sign=sg, delta=RUN.delta(inp, edit=(L, (sg * SIZE_L[L] * u).astype(np.float32)))))
    pd.DataFrame(srows).to_csv(sp2, index=False); maybe_push("steer"); el = time.time() - t0
    if (n_ + 1) % 4 == 0 or n_ == len(todo) - 1: print(f"  {n_ + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (n_ + 1) * (len(todo) - n_ - 1) / 60:.0f} min left", flush=True)
ST = pd.DataFrame(srows); maybe_push("steer done", force=True)
nat = (S["pd_k8"] - S["pd_k2"])[STEER_Q].mean(); say(f"\n[M3] natural count effect (k=8 minus k=2) on the {len(STEER_Q)} steering questions: {nat:+.3f} logits. Steering size = {STEER_SIZE} x |natural count shift of the state| at each layer (so a direction as strong as the natural count would give about {nat * STEER_SIZE:+.2f} if it were fully effective)")
rows = []
for L in STEER_L:
    A = AX[L]; Ephat = unit(A["Ep"]); wr = unit(RUN.readout_axis("yn")); eff, cE, cR = {}, {}, {}
    for nm, u in DIRS[L].items():
        x = ST[(ST.layer == L) & (ST.dir == nm)]; pv = x.pivot(index="qi", columns="sign", values="delta"); eff[nm] = float(((pv[1] - pv[-1]) / 2).mean()); cE[nm] = float(u @ Ephat); cR[nm] = float(u @ wr)
    cal = [k for k in eff if k == "Ep" or k.startswith("cal") or k.startswith("null")]
    xs, ys = np.array([cE[k] for k in cal]), np.array([eff[k] for k in cal]); slope = float((xs @ ys) / (xs @ xs)); res = {k: eff[k] - slope * cE[k] for k in eff}
    nr = np.array([res[k] for k in cal if k.startswith("null") or k.startswith("cal")]); sd = float(nr.std(ddof=1)) if len(nr) > 1 else float("nan")
    for k in eff: rows.append(dict(layer=L, dir=k, cos_Ep=cE[k], cos_readout=cR[k], effect=eff[k], predicted=slope * cE[k], residual=res[k], z=res[k] / sd if sd == sd and sd > 0 else np.nan))
    rows.append(dict(layer=L, dir="(slope, null sd)", cos_Ep=np.nan, cos_readout=np.nan, effect=slope, predicted=np.nan, residual=sd, z=np.nan))
M3 = pd.DataFrame(rows); M3.to_csv(f"{OUT}/m3_steering.csv", index=False)
say("effect = mean over questions of [delta(+d) - delta(-d)]/2 in logits (Yes - No); predicted = slope x cos(d, POPE evidence axis), slope fitted on Ep, the 3 calibration directions and the nulls; z = residual / sd of the calibration+null residuals")
for L in STEER_L: say(f"\n  layer {L}:"); say(M3[M3.layer == L][["dir", "cos_Ep", "cos_readout", "effect", "predicted", "residual", "z"]].round(3).to_string(index=False))
say("Reading: the 'residual' of G / Gperp (gratings count direction), P / Pperp (POPE count direction) and Prior / Priorperp (learned-prior axis) is the part of the steering effect NOT explained by overlap with the POPE evidence axis. A residual inside the null spread (|z| < 2) = no evidence for a count- or prior-specific effect.")

In [ ]:
open(f"{OUT}/F2b_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push("final", force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT):
        if f != "states.npz": zf.write(f"{OUT}/{f}", f)                       # the states file is large; it is also in the checkpoint dataset
print(f"zip written: {OUT}.zip (without states.npz)")